# Set up 
## Compute
- 17.3 LTS Scala 2.13 Spark 4.0.0
- Policy Personal Compute 
## Libraries
- org.neo4j:neo4j-connector-apache-spark_2.13:6.0.0_for_spark_4

For Shared Compute add org.neo4j:neo4j-connector-apache-spark_2.13:6.0.0_for_spark_4 to allow list


In [0]:
# Secrets should not be here 
USER_NAME="neo4j"
PASSWORD="will make example with sso in the future"
DATABASE_NAME="neo4j"
DBMS="neo4j+s://[database instance id goes here].databases.neo4j.io"

from pyspark.sql import SparkSession
spark = (
    SparkSession.builder
    .config("neo4j.url", DBMS)
    .config("neo4j.authentication.basic.username", USER_NAME)
    .config("neo4j.authentication.basic.password", PASSWORD)
    .config("neo4j.database", DATABASE_NAME)
    .getOrCreate()
)

## Patient nodes

Select data, shape it so it fits the nodes you want to create

In [0]:
patients = spark.sql(
    """
    select 
        Id as id,
        BIRTHDATE as birthdate,
        SSN as ssn,
        FIRST as first_name,
        LAST as last_name
    from haklof.default.patients
    """
)
patients.display()

id,birthdate,ssn,first_name,last_name
24ac5e6f-13c9-6809-372d-67963937f018,1979-02-06,999-42-1854,Brett333,Nikolaus26
cf5a5a29-e282-0e44-0f0b-d508b4d743f6,2003-10-16,999-70-8420,Rene434,Terry864
a31c82c1-3e97-abaa-3d02-4d23f5e04bf3,1984-09-23,999-91-1095,Shari363,Pfannerstill264
25a9a71e-d206-7821-44d8-10ffd53488c5,2005-09-21,999-46-1446,Micheal721,Jacobs452
e7de1ec5-efe5-70e3-c37b-f681ab2f1c46,1998-10-17,999-29-1525,Joesph309,Ledner144
abb08e7d-dce2-0341-99d9-290ee7034ff1,1944-08-27,999-24-5410,Yolande652,Kunde533
ea1ecbf7-db1a-de02-7af2-1c03db3c3a2e,2002-04-12,999-96-5542,Effie9,Gerlach374
afc6e441-4975-49ce-1bdc-d5ec08f5e3e9,2000-07-28,999-49-6389,Caleb651,Pfannerstill264
2f2bf8bc-723f-657c-b4f3-7cce4b06d0c9,2003-05-11,999-26-7812,Andrés117,Villaseñor332
74c65893-d309-bae5-12fc-a4b9970e99db,2010-09-14,999-54-7027,Rutha66,Farrell962


In [0]:
(
    patients.write.format("org.neo4j.spark.DataSource")
    .mode("Overwrite")
    .option("labels", ":Patient")
    .option("node.keys","id")
    .option("schema.optimization.node.keys", "KEY")
    .save()
)

## Encounter nodes

In [0]:
encounters = spark.sql(
    """
    select 
        Id as id,
        START as start,
        STOP as stop,
        ENCOUNTERCLASS as class,
        CODE as code,
        DESCRIPTION as description,
        BASE_ENCOUNTER_COST as base_cost,
        TOTAL_CLAIM_COST as total_claim_cost,
        PAYER_COVERAGE as payer_coverage
    from haklof.default.encounters
    """
)
encounters.display()

id,start,stop,class,code,description,base_cost,total_claim_cost,payer_coverage
21c4d69e-eaec-a11b-26a2-e04daff8d4d4,1997-04-01T00:58:25Z,1997-04-01T01:13:25Z,wellness,162673000,General examination of patient (procedure),129.16,1315.1,0.0
bedf3f72-e763-2129-af62-36345387d75b,2013-10-24T15:49:58Z,2013-10-24T16:04:58Z,wellness,410620009,Well child visit (procedure),129.16,269.68,0.0
995b6e91-87e3-6f29-0e46-3c5ed8a34e9e,1998-04-07T00:58:25Z,1998-04-07T01:13:25Z,wellness,162673000,General examination of patient (procedure),129.16,1406.01,0.0
55355953-3837-17e4-e1e9-6bdc6e7b4672,2001-04-10T00:58:25Z,2001-04-10T01:13:25Z,wellness,162673000,General examination of patient (procedure),129.16,1067.37,0.0
44b20799-9d47-a007-befa-0430827979a7,2004-04-13T00:58:25Z,2004-04-13T01:13:25Z,wellness,162673000,General examination of patient (procedure),129.16,786.33,0.0
3c1603ad-6de8-fa74-ba34-4c03881a2e90,2007-04-17T00:58:25Z,2007-04-17T01:13:25Z,wellness,162673000,General examination of patient (procedure),129.16,1292.78,294.58
82dd71fb-a9df-bd6d-73db-0d0134fc9647,2010-12-01T11:58:25Z,2010-12-01T12:13:25Z,ambulatory,185345009,Encounter for symptom,77.49,77.49,0.0
74a79ca1-fb80-c4b5-2f0b-84dc001013f9,2013-04-23T00:58:25Z,2013-04-23T01:13:25Z,wellness,162673000,General examination of patient (procedure),129.16,786.33,0.0
0d407517-124a-7fd3-a276-8187798bead1,2014-10-30T15:49:58Z,2014-10-30T16:04:58Z,wellness,410620009,Well child visit (procedure),129.16,691.24,0.0
5b0b1625-4048-b86a-2713-9ea59788827f,2016-04-26T00:58:25Z,2016-04-26T01:13:25Z,wellness,162673000,General examination of patient (procedure),129.16,786.33,0.0


In [0]:
(
    encounters.write.format("org.neo4j.spark.DataSource")
    .mode("Overwrite")
    .option("labels", ":Encounter")
    .option("node.keys","id")
    .option("schema.optimization.node.keys", "KEY")
    .save()
)

## Create relationship (Patient) -[:HAS_ENCOUNTER]-> (:Encounter)

For relationships, name the from node - to node keys in a clever way. Select desired additional relationship properties and name them well.

In [0]:
patient_encounters = spark.sql(
    """
    select 
        Id as encounter_id,
        PATIENT as patient_id
    from haklof.default.encounters
    """
)
patient_encounters.display()

encounter_id,patient_id
21c4d69e-eaec-a11b-26a2-e04daff8d4d4,24ac5e6f-13c9-6809-372d-67963937f018
bedf3f72-e763-2129-af62-36345387d75b,cf5a5a29-e282-0e44-0f0b-d508b4d743f6
995b6e91-87e3-6f29-0e46-3c5ed8a34e9e,24ac5e6f-13c9-6809-372d-67963937f018
55355953-3837-17e4-e1e9-6bdc6e7b4672,24ac5e6f-13c9-6809-372d-67963937f018
44b20799-9d47-a007-befa-0430827979a7,24ac5e6f-13c9-6809-372d-67963937f018
3c1603ad-6de8-fa74-ba34-4c03881a2e90,24ac5e6f-13c9-6809-372d-67963937f018
82dd71fb-a9df-bd6d-73db-0d0134fc9647,24ac5e6f-13c9-6809-372d-67963937f018
74a79ca1-fb80-c4b5-2f0b-84dc001013f9,24ac5e6f-13c9-6809-372d-67963937f018
0d407517-124a-7fd3-a276-8187798bead1,cf5a5a29-e282-0e44-0f0b-d508b4d743f6
5b0b1625-4048-b86a-2713-9ea59788827f,24ac5e6f-13c9-6809-372d-67963937f018


In [0]:
(
    patient_encounters.repartition(1).write.mode("Overwrite").format("org.neo4j.spark.DataSource")
    .option("relationship", "HAS_ENCOUNTER")
    .option("relationship.save.strategy", "keys")
    .option("relationship.source.save.mode", "Match")
    .option("relationship.source.labels", ":Patient")
    .option("relationship.source.node.keys", "patient_id:id") # patient_id:id  [column in the dataframe]:[property used as nodekey]
    .option("relationship.target.save.mode", "Match")
    .option("relationship.target.labels", ":Encounter")
    .option("relationship.target.node.keys", "encounter_id:id")
    .save()
)

## Allergies

In [0]:
patient_allergies = spark.sql(
    """
    select 
        CODE as code,
        SYSTEM as system,
        PATIENT as patient_id,
        DESCRIPTION as description,
        TYPE as type,
        CATEGORY as category
    from haklof.default.allergies
    """
)
patient_allergies.display()

code,system,patient_id,description,type,category
735029006,SNOMED-CT,abb08e7d-dce2-0341-99d9-290ee7034ff1,Shellfish (substance),allergy,food
288328004,SNOMED-CT,ea1ecbf7-db1a-de02-7af2-1c03db3c3a2e,Bee venom (substance),allergy,environment
84489001,SNOMED-CT,ea1ecbf7-db1a-de02-7af2-1c03db3c3a2e,Mold (organism),allergy,environment
260147004,SNOMED-CT,ea1ecbf7-db1a-de02-7af2-1c03db3c3a2e,House dust mite (organism),allergy,environment
264287008,SNOMED-CT,ea1ecbf7-db1a-de02-7af2-1c03db3c3a2e,Animal dander (substance),allergy,environment
256277009,SNOMED-CT,ea1ecbf7-db1a-de02-7af2-1c03db3c3a2e,Grass pollen (substance),allergy,environment
782576004,SNOMED-CT,ea1ecbf7-db1a-de02-7af2-1c03db3c3a2e,Tree pollen (substance),allergy,environment
5640,RxNorm,ea1ecbf7-db1a-de02-7af2-1c03db3c3a2e,Ibuprofen,allergy,medication
102263004,SNOMED-CT,ea1ecbf7-db1a-de02-7af2-1c03db3c3a2e,Eggs (edible) (substance),allergy,food
29046,RxNorm,74227cb9-3bc7-2bd5-ee74-7a011136056e,Lisinopril,intolerance,medication


In [0]:
write_allergy_query="""
  merge (a:Allergy{code: event.code, system: event.system})
  on create set a.description = event.description,
                a.type = event.type,
                a.category = event.category
  merge (p:Patient{id:event.patient_id})
  merge (p)-[:HAS_ALLERGY]->(a)
"""
(
  patient_allergies.write
  .format("org.neo4j.spark.DataSource")
  .option("query", write_allergy_query)
  .option("script","CREATE CONSTRAINT IF NOT EXISTS FOR (n:Allergy) REQUIRE (n.code, n.system) IS NODE KEY;")\
  .mode("Overwrite")
  .save()
)

In [0]:
patient_allergies_vectors = spark.sql(
    """
    select 
        CODE as code,
        SYSTEM as system,
        `__db_DESCRIPTION_vector` as description_vector
    from haklof.default.allergies_vectors_writeback_table
    """
)
patient_allergies_vectors.display()

code system description_vector 111088007 SNOMED-CT List(-0.040496826171875, 0.0255279541015625, -0.0028858184814453125, 0.0179443359375, -0.0172271728515625, -0.03973388671875, -0.021148681640625, 0.0016241073608398438, 0.020538330078125, -0.0243377685546875, -0.033172607421875, 0.0136871337890625, -0.019622802734375, -0.0256195068359375, 0.0225982666015625, 0.0218963623046875, 0.0311737060546875, -0.005397796630859375, -0.038909912109375, -0.0220794677734375, -0.01442718505859375, -0.0279083251953125, -0.01082611083984375, 0.0299530029296875, 0.0298004150390625, -0.0153656005859375, -0.0078277587890625, -0.06103515625, 0.03997802734375, 0.01496124267578125, 0.051116943359375, -0.05267333984375, 0.05706787109375, 0.00907135009765625, -0.00894927978515625, 0.0274810791015625, 0.067626953125, 0.0186004638671875, 0.00875091552734375, 0.0029392242431640625, -0.009307861328125, -0.017364501953125, -0.036834716796875, 0.006221771240234375, 0.00901031494140625, 0.0335693359375, -0.04254150390625, 0.00402069091796875, 0.045013427734375, 0.0056915283203125, -0.012542724609375, -0.0194549560546875, 0.030517578125, 0.067626953125, 5.688667297363281E-4, -0.00388336181640625, -0.0243377685546875, 0.06829833984375, 0.024627685546875, -0.0374755859375, -0.0277252197265625, -0.01403045654296875, -0.0162353515625, -0.0281524658203125, -0.0498046875, -0.018829345703125, -0.07183837890625, -0.0160369873046875, -0.0037403106689453125, -0.006015777587890625, 0.038848876953125, 0.0277252197265625, -0.07177734375, 0.0235595703125, -0.07672119140625, -0.047119140625, -5.588531494140625E-4, 0.032073974609375, -0.03656005859375, 0.053497314453125, 0.03631591796875, 0.0222930908203125, -0.02459716796875, -0.0034618377685546875, 0.00492095947265625, -0.006465911865234375, -0.02301025390625, 0.037811279296875, -0.0280914306640625, -0.034698486328125, -0.016632080078125, -0.0214080810546875, -0.05120849609375, 0.01174163818359375, 0.0123443603515625, -0.0011968612670898438, -0.0462646484375, -0.0204315185546875, -0.01995849609375, 0.025390625, 0.005542755126953125, 0.0025234222412109375, 0.01428985595703125, -0.05194091796875, -0.003589630126953125, 0.0355224609375, -0.0048980712890625, -0.01177215576171875, -0.0196990966796875, -0.009246826171875, 0.0142364501953125, -0.0176544189453125, -0.01543426513671875, 0.0214691162109375, 0.04058837890625, 0.006458282470703125, 0.020751953125, -1.475811004638672E-4, -0.039337158203125, -0.00403594970703125, 0.0257720947265625, 0.01047515869140625, 0.005035400390625, -0.023193359375, -0.003936767578125, 0.01006317138671875, 0.003948211669921875, -0.01317596435546875, 0.016632080078125, 0.02435302734375, 0.0380859375, -0.046051025390625, -0.02239990234375, -0.031585693359375, -0.00814056396484375, 0.01473236083984375, 0.006374359130859375, 0.0027217864990234375, -0.038543701171875, -0.0214691162109375, 0.002162933349609375, -0.007476806640625, 0.045989990234375, 0.03924560546875, -0.0158538818359375, 0.0182037353515625, -0.03240966796875, 0.0016384124755859375, -0.03326416015625, 0.03106689453125, -0.03277587890625, 0.04534912109375, 0.01146697998046875, 0.06488037109375, -0.0188446044921875, -0.089599609375, 0.01617431640625, 0.0333251953125, 0.013916015625, -0.028961181640625, 0.005634307861328125, 0.0083160400390625, 0.05731201171875, -0.033599853515625, 0.0255889892578125, 0.0133514404296875, 0.028472900390625, -0.01006317138671875, -0.01293182373046875, -7.143020629882812E-4, 0.0021343231201171875, -0.0044708251953125, -0.01068115234375, -0.0124969482421875, -0.04547119140625, -0.031982421875, 0.02667236328125, 0.045654296875, 0.0034942626953125, 0.023681640625, 0.0274810791015625, 0.08465576171875, -0.01100921630859375, 0.0283050537109375, -0.01372528076171875, -0.06396484375, 0.01543426513671875, 0.037384033203125, 7.610321044921875E-4, 0.018524169921875, -0.0219573974609375, -0.0120697021484375, 0.048065185546875, 0.01332855224609375, -0.04766845703125, -0.0094146728515625, 0.018218994140625, 0.01756286621093

In [0]:
write_allergy_vectors_query = """
  merge (a:Allergy{code: event.code, system: event.system})
  set a.description_vector = vector(
      event.description_vector,
      1536,
      FLOAT
    )
"""

(
  patient_allergies_vectors.write
  .format("org.neo4j.spark.DataSource")
  .option("query", write_allergy_vectors_query)
  .option("script","CREATE VECTOR INDEX allergy_description IF NOT EXISTS FOR (n:Allergy) ON (n.description_vector) with [n.system];")\
  .mode("Overwrite")
  .save()
)